# Normalize the following tensors for training
- Position
- Velocity
- Frame size

During the normalization, position and velocity tensors calculate the related stats without using the fillers. \
For frame size, filler value (sentinel) is 0, the function ignores any value of 0. \
Similarly for the position, the filler is 0. During the calculation of stats, any vector of [0.0,0.0,0.0] is ignored. 

In [1]:
# Import libraries
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
import torch
from torch.utils.data import DataLoader
import time 
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from math import cos, pi
from torch.optim.lr_scheduler import LambdaLR
import os
import pickle

# Configuration Settings

In [2]:
from pathlib import Path

device = "cpu"

seed_identifier = "seed2"
carla_run_identifier = "separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy"
# network_configuration = "bw10mbit_delay0_jitter0_loss0"
network_configuration = "bw50mbit_delay20_jitter5_loss0"

tensors_path = Path(f"/home/gaofeng/zanoria/grayassets_datasets/{seed_identifier}/{carla_run_identifier}/tensors/")


exp_meta_tensor = torch.load(tensors_path/"meta_data_tensor.pt", map_location=device)  # [E,8] = [brand_id, color_id, yaw, vx, vy, x, y, rows]
positions_tensor     = torch.load(tensors_path/"position_gt_tensor.pt",     map_location=device)  # [E,G*T,3] 
velocities_tensor    = torch.load(tensors_path/"velocity_gt_tensor.pt",    map_location=device)  # [E,G*T,3]
frame_sizes     = torch.load(tensors_path/"frame_size_tensor.pt",     map_location=device)  # [E,G*T,4]
packet_sizes    = torch.load(tensors_path/"packetsizes"/network_configuration/"windowed_packet_size_tensor.pt",     map_location=device)  # [E,G*T,4]
fov_gt_tensor = torch.load(tensors_path/"in_n_out_gt_tensor.pt",     map_location=device)  # [E,5,G*T] (i,0) binary for whole sequence, (i,j) 0/1/2

# DROP THE FIRST GOP FROM EACH
positions_tensor = positions_tensor[:,10:,:]
velocities_tensor = velocities_tensor[:,10:,:]
frame_sizes = frame_sizes[:,10:,:]
# DROP THE FIRST (MISSHAPEN) GOP, AND THE FINAL PADDING ELEMENT
packet_sizes = packet_sizes[:,9:-1,:]
fov_gt_tensor = fov_gt_tensor[:,:,10:]

# shape [E,T_max]
fov_all_tensor = fov_gt_tensor[:,0,:] # tensor for keeping binary FOV for the sequence (this is used for position estimation)
# shape [E,N,T_max]
fov_cameras_tensor = fov_gt_tensor[:,1:,:] # tensor for individual cameras 0/1/2

In [3]:
# Parameters
if seed_identifier == "seed1":
    E = 1010 # Number of total videos
elif seed_identifier == "seed2":
    E = 1008 # Number of total videos
# G = 48 # GOP per video
G = 47 # DROP THE FIRST GOP
N = 4 # Number of nodes
T = 10 # GOP length
W = 3 # Number of GOPs in each window
D = 120 # Hidden dimension of the transformer
detach_every = 3 # Parameter for gradient flow over windows

In [4]:
# shape of the tensors
print(exp_meta_tensor.shape)
print(positions_tensor.shape)
print(velocities_tensor.shape)
print(frame_sizes.shape)
print(packet_sizes.shape)
print(fov_gt_tensor.shape)
print(fov_all_tensor.shape)
print(fov_cameras_tensor.shape)


torch.Size([1008, 8])
torch.Size([1008, 470, 3])
torch.Size([1008, 470, 3])
torch.Size([1008, 470, 4])
torch.Size([1008, 470, 4])
torch.Size([1008, 5, 470])
torch.Size([1008, 470])
torch.Size([1008, 4, 470])


In [5]:
# Normalize the frame sizes to 0 mean, unit variance per channel in each GoP

# IGNORES ANY VALUE EQUAL TO THE SENTINEL WHILE CALCULATING THE NORMALIZATION PARAMETERS
def compute_stats_and_normalize(
    frames: torch.Tensor,          # shape [E, G, T, N] e.g., [2018, 48, 10, 4]
    sentinel: float = 0.0,
    unbiased: bool = False,        # False = population std, True = sample std
    eps: float = 1e-8              # numerical stability for std
):
    """
    Returns:
      mean_kl: [T, N]
      std_kl:  [T, N]
      norm:    [E, G, T, N] normalized, with invalid entries kept as `sentinel`
      counts:  [T, N] number of valid entries used per (k,l)
    """
    assert frames.ndim == 4, "Expected [E, G, K, L]"
    E, G, K, L = frames.shape

    x = frames.float()
    mask = x.ne(sentinel)                # True where valid

    # counts per (k,l), summed over (E,G)
    counts = mask.sum(dim=(0, 1))        # [K, L]

    # means per (k,l)
    sum_valid = (x * mask).sum(dim=(0, 1))             # [K, L]
    mean_kl = sum_valid / counts.clamp_min(1)          # avoid /0; mean unused where count=0

    # std per (k,l)
    # broadcast mean over (E,G)
    mean_b = mean_kl.view(1, 1, K, L)
    diff = (x - mean_b)
    diff2 = (diff * mask) ** 2
    ssq = diff2.sum(dim=(0, 1))                        # [K, L]
    denom = (counts - 1) if unbiased else counts
    std_kl = torch.sqrt((ssq / denom.clamp_min(1)).clamp_min(0.0))  # [K, L]

    # normalize valid entries; keep sentinel for invalid
    std_b = std_kl.view(1, 1, K, L)
    norm = torch.full_like(x, fill_value=sentinel)
    valid_norm = diff / std_b.clamp_min(eps)
    norm = torch.where(mask, valid_norm, norm)

    return mean_kl, std_kl, norm, counts

mean_kl, std_kl, normalized_frame_sizes, counts = compute_stats_and_normalize(frame_sizes.reshape(E, G, T, N), sentinel=0.0, unbiased=False)

# Quick sanity prints:
print(mean_kl.shape, std_kl.shape, normalized_frame_sizes.shape)
print(mean_kl)
print(std_kl)


torch.Size([10, 4]) torch.Size([10, 4]) torch.Size([1008, 47, 10, 4])
tensor([[225851.4844, 232858.0781, 232395.1719, 233810.9844],
        [  2591.4475,   8830.7314,   4410.1187,   9031.4629],
        [  3300.3577,  10697.8926,   5574.6846,  10190.7910],
        [  4073.2996,  12732.4375,   7281.7520,  13384.3574],
        [  4632.2612,  13700.3223,   7977.0317,  14403.7988],
        [  5573.6528,  14618.2812,   9167.0117,  15887.2520],
        [  5628.2153,  14899.8379,   9232.2100,  15997.9717],
        [  5402.8662,  14805.7031,   9810.4912,  15907.4609],
        [  6001.6724,  14954.4521,  10216.0146,  16025.5723],
        [  6113.9712,  15003.6055,  10405.3330,  16249.8750]])
tensor([[9295.2773, 8776.3438, 3721.9045, 9122.5859],
        [1037.9866, 1012.2521,  810.0974,  873.4919],
        [1239.8440, 1382.6263,  915.7369, 1130.8077],
        [1388.5912,  988.1290, 1425.8311, 1216.7189],
        [1374.5344, 1332.5762, 1497.1119, 1189.1488],
        [1256.0308, 1468.1471, 1213.100

In [6]:
# Normalize the packet sizes to 0 mean, unit variance per channel in each GoP

mean_kl, std_kl, normalized_packet_sizes, counts = compute_stats_and_normalize(packet_sizes.reshape(E, G, T, N), sentinel=0.0, unbiased=False)

# Quick sanity prints:
print(mean_kl.shape, std_kl.shape, normalized_packet_sizes.shape)
print(mean_kl)
print(std_kl)


torch.Size([10, 4]) torch.Size([10, 4]) torch.Size([1008, 47, 10, 4])
tensor([[164240.1094, 165582.6562, 165417.0312, 165586.8906],
        [ 72962.4766,  85481.3438,  80496.1094,  86657.3359],
        [  3607.1506,  11084.3516,   5889.8584,  10597.5928],
        [  4357.6260,  13289.1816,   7601.8091,  13935.4072],
        [  4885.2271,  14237.1924,   8357.0986,  14988.5947],
        [  5909.5591,  15183.7822,   9524.2119,  16478.6172],
        [  5935.1060,  15491.7490,   9618.7314,  16661.0059],
        [  5669.4448,  15405.7988,  10240.8184,  16527.9082],
        [  6329.6475,  15534.3125,  10622.8721,  16655.8770],
        [  6410.2793,  15575.1621,  10806.3896,  16889.5918]])
tensor([[20336.1953, 19539.4219, 20102.0820, 19550.0430],
        [22757.3438, 21712.6680, 20614.6992, 22308.2324],
        [ 1576.4597,  2076.0039,  1572.8586,  1862.5698],
        [ 1797.5038,  1760.5975,  1991.1302,  1941.2521],
        [ 1770.8755,  2049.4211,  2048.4504,  1930.9934],
        [ 1743.3896

In [7]:
# Compute the normalization stats for the position tensor

# Any time step with position [0,0,0] is excluded from the normalization process (for calculation of )

def compute_position_norm_stats(
    positions_tensor: torch.Tensor,  # [E, 480, 3]
    unbiased: bool = False,          # False = population std, True = sample std (Bessel's correction)
    eps: float = 1e-12               # numerical floor
):
    assert positions_tensor.ndim == 3 and positions_tensor.shape[-1] == 3, \
        f"Expected [E, T, 3], got {tuple(positions_tensor.shape)}"

    x = positions_tensor.float()                      # [E, T, 3]
    # Mask valid timesteps: exclude filler rows exactly equal to [0,0,0]
    valid_mask = (x.abs().sum(dim=-1) > 0)           # [E, T] True where valid
    count_valid = int(valid_mask.sum().item())
    if count_valid == 0:
        raise ValueError("No valid (non-[0,0,0]) positions found.")

    # Compute mean over valid entries (same count for all 3 dims since mask is row-wise)
    sum_vec = (x * valid_mask.unsqueeze(-1)).sum(dim=(0, 1))      # [3]
    mean_xyz = sum_vec / max(count_valid, 1)                       # [3]

    # Compute std over valid entries
    diff = x - mean_xyz.view(1, 1, 3)
    ssq = ((diff ** 2) * valid_mask.unsqueeze(-1)).sum(dim=(0, 1)) # [3]
    denom = (count_valid - 1) if unbiased else count_valid
    std_xyz = torch.sqrt(torch.clamp(ssq / max(denom, 1), min=0.0) + eps)  # [3]

    return mean_xyz, std_xyz, count_valid


# Normalize using the calculated stats

def normalize_positions(
    positions_tensor: torch.Tensor,   # [E, T, 3]
    mean_xyz: torch.Tensor,           # [3]
    std_xyz: torch.Tensor,            # [3]
    keep_fillers_zero: bool = True,
    eps: float = 1e-12
) -> torch.Tensor:
    """
    Standardize valid rows: (pos - mean_xyz) / std_xyz, per-axis.
    Rows equal to [0,0,0] are treated as fillers and excluded from normalization.
    If keep_fillers_zero=True, fillers remain [0,0,0] in the output; otherwise they get standardized too.

    Returns:
      positions_norm: [E, T, 3]
    """
    assert positions_tensor.ndim == 3 and positions_tensor.shape[-1] == 3, \
        f"Expected [E, T, 3], got {tuple(positions_tensor.shape)}"
    assert mean_xyz.shape == (3,) and std_xyz.shape == (3,), "mean/std must be shape [3]"

    x = positions_tensor.to(dtype=torch.float32)
    mean = mean_xyz.to(x.device, x.dtype).view(1, 1, 3)
    std  = std_xyz.to(x.device, x.dtype).clamp_min(eps).view(1, 1, 3)

    # Valid rows are anything not exactly [0,0,0]
    valid_mask = (x.abs().sum(dim=-1) > 0)   # [E, T]

    z = (x - mean) / std                     # standardized everywhere

    if keep_fillers_zero:
        out = torch.zeros_like(x)
        out[valid_mask] = z[valid_mask]
    else:
        out = z
    return out

mean_xyz, std_xyz, _ = compute_position_norm_stats(positions_tensor)  
normalized_position_tensor = normalize_positions(positions_tensor, mean_xyz, std_xyz, keep_fillers_zero=True)
print(normalized_position_tensor.shape)  # -> (E, 480, 3)
print("Mean:", mean_xyz.tolist())
print("Std :", std_xyz.tolist())


torch.Size([1008, 470, 3])
Mean: [-51.025848388671875, 22.814016342163086, 0.022433748468756676]
Std : [19.927133560180664, 20.020418167114258, 0.04965436831116676]


In [8]:
# Normalize the velocities as well

# THIS ASSUMES CONSTANT VELOCITY
# THERE IS NO FILLER HERE
def compute_velocity_norm_stats(
    velocities: torch.Tensor,  # [E, 3]
    unbiased: bool = False,    # False=population std, True=sample std
):
    assert velocities.ndim == 2 and velocities.shape[1] == 3, \
        f"Expected [E,3], got {tuple(velocities.shape)}"
    v = velocities.float()
    mean_3 = v.mean(dim=0)                              # [3]
    std_3  = v.std(dim=0, unbiased=unbiased)            # [3]
    return mean_3, std_3

def normalize_velocities(
    velocities: torch.Tensor,  # [E, 3]
    mean_3: torch.Tensor,      # [3]
    std_3: torch.Tensor,       # [3]
    eps: float = 1e-12
) -> torch.Tensor:
    v = velocities.float()
    mean_3 = mean_3.to(v.dtype).view(1, 3)
    std_3  = std_3.to(v.dtype).clamp_min(eps).view(1, 3)
    return (v - mean_3) / std_3                         # [E,3]

mean_3, std_3 = compute_velocity_norm_stats(velocities_tensor[:,25,:], unbiased=False)
normalized_velocity_tensor = normalize_velocities(velocities_tensor[:,25,:], mean_3, std_3)
print(mean_3, std_3, normalized_velocity_tensor.shape)


tensor([ 0.0478, -0.2856, -0.1407]) tensor([9.2621e+00, 9.4066e+00, 3.2004e-03]) torch.Size([1008, 3])


In [9]:
# save the extracted tensors

# output_directory = "tensors/seed2"
# output_directory = main_dir
# os.makedirs(output_directory, exist_ok=True)

filename = "normalized_frames_tensor.pt"
# Construct the full file path
# file_path = os.path.join(output_directory, filename)
file_path = tensors_path / filename
torch.save(normalized_frame_sizes, file_path)

filename = f"normalized_windowed_packets_tensor.pt"
# Construct the full file path
# file_path = os.path.join(output_directory, filename)
file_path = tensors_path / "packetsizes" / network_configuration / filename
torch.save(normalized_packet_sizes, file_path)

filename = "normalized_position_tensor.pt"
# file_path = os.path.join(output_directory, filename)
file_path = tensors_path / filename
torch.save(normalized_position_tensor, file_path)

filename = "normalized_velocity_tensor.pt"
# file_path = os.path.join(output_directory, filename)
file_path = tensors_path / filename
torch.save(normalized_velocity_tensor, file_path)